Take Home Exercise #2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import count, avg, min, max, sum
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from pyspark.sql.functions import when, upper, substring
from pyspark.sql.functions import datediff
from pyspark.sql.functions import desc

In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv", header = True, inferSchema = True)
display(df_laptimes)

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header = True, inferSchema = True)
display(df_drivers)

In [0]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header = True, inferSchema = True).orderBy("raceId", "driverId")
display(df_pitstops)

1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

To answer this question, I used the `pit_stops` dataset because it contains pit stop records for each driver in each race. I first grouped the data by `raceId` and `driverId` to calculate the average pit stop time in milliseconds.

Then, to identify the fastest and slowest pit stops together with their corresponding stop number and lap number, I ranked pit stop records within each race-driver group based on pit stop duration. I selected the smallest value as the fastest pit stop and the largest value as the slowest pit stop.

Finally, I joined these results with the `drivers` dataset so the final output includes driver information, which provides a clearer insights.

In [0]:
df_drivers_clean = df_drivers.drop("url")

# average pit stop time for each driver in each race
df_avg_pit = df_pitstops.groupBy("raceId", "driverId").agg(avg("milliseconds").alias("avg_pit_milliseconds"))

#fastest pit stop row for each driver in each race
fastest_window = Window.partitionBy("raceId", "driverId").orderBy(col("milliseconds").asc())

df_fastest = df_pitstops.withColumn("rn", row_number().over(fastest_window)).filter(col("rn") == 1).select("raceId","driverId", col("milliseconds").alias("fastest_pit_milliseconds"), col("stop").alias("fastest_stop"), col("lap").alias("fastest_lap"))

# slowest pit stop row for each driver in each race
slowest_window = Window.partitionBy("raceId", "driverId").orderBy(col("milliseconds").desc())

df_slowest = df_pitstops.withColumn("rn", row_number().over(slowest_window)).filter(col("rn") == 1).select("raceId","driverId", col("milliseconds").alias("slowest_pit_milliseconds"), col("stop").alias("slowest_stop"), col("lap").alias("slowest_lap"))

#Combine
df_pit_final = df_avg_pit.join(df_fastest, on=["raceId", "driverId"], how="left").join(df_slowest, on=["raceId", "driverId"], how="left").join(df_drivers_clean, on="driverId", how="left").select("driverId","driverRef", "number", "code", "forename", "surname", "dob", "nationality","raceId", "avg_pit_milliseconds", "fastest_pit_milliseconds", "fastest_stop", "fastest_lap", "slowest_pit_milliseconds", "slowest_stop","slowest_lap").orderBy("raceId", "driverId")

display(df_pit_final)

By using `drop`, I deleted the url column of drivers dataset because I think it's useless for data analysis, and the sheet could become cleaner after dropping it.

Then, I used `.groupBy("raceId", "driverId")` and `avg("milliseconds")` to calculate the average pit stop time for each driver in each race. By using `allias`, I can rename the certain column, so it will be clear when we output the final result.

To find the fastest and slowest pit stop together with the related `stop` and `lap`, I used window functions. `Window.partitionBy("raceId", "driverId")` creates groups for each driver in each race. Then I used `orderBy(col("milliseconds").asc())` to rank pit stops from fastest to slowest, and `orderBy(col("milliseconds").desc())` to rank pit stops from slowest to fastest.

Next, `row_number()` assigns an order within each group, and filtering for `rn == 1` keeps only the fastest or slowest row. This allows me to keep the associated `stop` and `lap` values, which cannot be obtained from a simple aggregation alone.

Finally, I joined the average, fastest, and slowest pit stop results together and merged them with the drivers dataset using `driverId`.

2. Rank order by finishing position the average time spent at the pit stop in each race.

To answer this question, I can directly use the average pit stop time for each driver from last question data `df_avg_pit`. 

Then, I joined this aggregated table with the `results` dataset to obtain each driver's finishing position. I used an inner join so that only drivers with available pit stop data are included in the final ranking. Finally, I joined the `drivers` dataset to add driver names and sorted the table by `raceId` and `positionOrder`.

In [0]:
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
display(df_results)

In [0]:
#join with results to get finishing position
df_position_ranking = df_results.join(df_avg_pit, on=["raceId", "driverId"], how="inner").join(df_drivers_clean, on="driverId", how="left").select("raceId", "positionOrder", "driverId", "code", "forename", "surname", "avg_pit_milliseconds").orderBy("raceId", "positionOrder")

display(df_position_ranking)

I first used the average pit stop time for each driver from last question data `df_avg_pit`

Next, I joined this aggregated pit stop table with the `results` dataset using `raceId` and `driverId`, because these columns identify the same driver in the same race across both datasets. I used an inner join so that only records with pit stop data are kept, which avoids null values in the average pit stop column.

Then, I joined the `drivers` dataset using `driverId` to include driver names and codes.

Finally, I sorted the result using `.orderBy("raceId", "positionOrder")` so that drivers are ranked by finishing position within each race.

3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

The missing values in the `code` column from drivers dataset are represented as "\N". I will directly replace these values.

I will generate a new code using the first three letters of the driver's surname in uppercase. For all other rows, I kept the original code unchanged.

In [0]:
df_drivers_filled = df_drivers.withColumn("code", when(col("code") == "\\N", upper(substring(col("surname"), 1, 3))).otherwise(col("code")))

display(df_drivers_filled)

I used `.withColumn()` together with `when()` to update the `code` column.

The condition `col("code") == "\\N"` identifies rows where the code is missing. For those rows, I applied `upper(substring(col("surname"), 1, 3))` to create a new three-letter code from the driver's surname.

The `.otherwise(col("code"))` ensures that existing valid codes remain unchanged.

4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

I first defined age as the driver’s age on the date of the race. I calculated it as the difference between the race date and the driver’s date of birth, measured in years.

Then, I joined the `results`, `drivers`, and `races` datasets so that each row represents a driver in a specific race and contains both the driver’s date of birth and the race date.

After creating the `Age` column, I identified the youngest and oldest driver in each race. To do this, I ranked drivers within each `raceId` by age in ascending order to find the youngest driver, and in descending order to find the oldest driver.

Finally, I joined the youngest-driver table and the oldest-driver table by `raceId` so that the final result shows both the youngest and oldest driver for each race.

In [0]:
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
display(df_races)

In [0]:
df_age = df_results.join(df_drivers_clean, on="driverId", how="left").join(df_races, on="raceId", how="left")

df_age = df_age.withColumn("Age", datediff(col("date"), col("dob")) / 365).select("raceId", "driverId", "forename", "surname", "dob", "Age", "date", "name", "positionOrder")

display(df_age)

In [0]:
#youngest part
youngest_window = Window.partitionBy("raceId").orderBy(col("Age").asc())

df_youngest = df_age.withColumn("rn", row_number().over(youngest_window)).filter(col("rn") == 1).select("raceId", "name", "date", "driverId", "forename", "surname", col("Age").alias("youngest_age"))

# oldest part
oldest_window = Window.partitionBy("raceId").orderBy(col("Age").desc())

df_oldest = df_age.withColumn("rn", row_number().over(oldest_window)).filter(col("rn") == 1).select("raceId", "driverId", "forename", "surname", col("Age").alias("oldest_age"))

# combine
df_age_final = df_youngest.join(df_oldest, on="raceId")

display(df_age_final)

First, I used `.join()` to combine the `results`, `drivers`, and `races` datasets by using `driverId` and `raceId`. Each rowcan  contain the driver information and the corresponding race date.

Next, I created a new column called `Age` by using:

`datediff(col("date"), col("dob")) / 365`

The `datediff()` function calculates the number of days between the race date and the driver’s date of birth, and dividing by 365 converts that value into years. This is my definition of age for this assignment.

To find the youngest driver in each race, I created a window with:

`Window.partitionBy("raceId").orderBy(col("Age").asc())`

`partitionBy("raceId")` groups the data by race, and `orderBy(col("Age").asc())` sorts drivers from youngest to oldest within each race. Then, `row_number()` assigns a rank to each driver in that race, and filtering with `col("rn") == 1` keeps only the youngest driver.

I used the same logic for the oldest driver, but changed the sorting order to descending with:

`orderBy(col("Age").desc())`

This ranks drivers from oldest to youngest, so again `row_number() == 1` identifies the oldest driver in each race.

Finally, I used `.alias()` inside `.select()` to rename the output columns, such as `youngest_age` and `oldest_age`, so the final table is easier to read. Then I joined the youngest and oldest tables on `raceId` to produce the final result.

5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

I will use the `results` dataset to identify each driver's finishing position in every race. Since the question asks for the number of podium finishes at any given race, I defined this as the cumulative number of podium finishes up to and including that race.

First, I will create three indicator columns:
- `win` for 1st place finishes
- `second_place` for 2nd place finishes
- `third_place` for 3rd place finishes

Each column takes the value 1 if the condition is met and 0 otherwise.

Then, I will calculate cumulative totals for each driver across races in chronological order. This gives the number of wins, second places, and third places that each driver has achieved up to each race.

Finally, I will sort the results by the number of wins in descending order, and then by race date in ascending order, so that drivers with more wins appear first while preserving the timeline of races.

In [0]:
df_races_useful = df_races.select("raceId", "name", "year", "round", "date")
df_podium = df_results.join(df_races_useful, on="raceId", how="left").select("raceId", "driverId", "positionOrder", "date")
display(df_podium)

In [0]:
df_podium_rank = df_podium.withColumn("win", when(col("positionOrder") == 1, 1).otherwise(0))\
                    .withColumn("second_place", when(col("positionOrder") == 2, 1).otherwise(0))\
                    .withColumn("third_place", when(col("positionOrder") == 3, 1).otherwise(0))

# Calculate totals for each driver over time
podium_window = Window.partitionBy("driverId").orderBy("date", "raceId").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_podium = df_podium_rank.withColumn("wins", sum("win").over(podium_window))\
                    .withColumn("second_places", sum("second_place").over(podium_window))\
                    .withColumn("third_places", sum("third_place").over(podium_window))

# Add driver names and keep relevant columns
df_podium_final = df_podium.join(df_drivers_clean, on="driverId", how="left").select("raceId", "date","driverId", "forename", "surname", "positionOrder", "wins", "second_places", "third_places").orderBy(col("wins").desc(), col("date").asc())

display(df_podium_final)

In [0]:
df_podium = df_podium.select("raceId", "driverId", "positionOrder", "date")

df_podium = df_podium.withColumn(
    "win",
    when(col("positionOrder") == 1, 1).otherwise(0)
).withColumn(
    "second_place",
    when(col("positionOrder") == 2, 1).otherwise(0)
).withColumn(
    "third_place",
    when(col("positionOrder") == 3, 1).otherwise(0)
)

display(df_podium.filter(col("win") == 1))

I joined the `results` dataset with the `races` dataset using `raceId` so that each row includes the race date. This is necessary to ensure that the cumulative calculations follow the correct chronological order.

Next, I created three indicator columns using `.withColumn()` and `when()`:
- `win` is 1 when `positionOrder == 1`
- `second_place` is 1 when `positionOrder == 2`
- `third_place` is 1 when `positionOrder == 3`

These columns identify whether a driver achieved a podium position in each race.

Then, I defined a window using:
`Window.partitionBy("driverId").orderBy("date", "raceId").rowsBetween(Window.unboundedPreceding, Window.currentRow)`

`partitionBy("driverId")` ensures that each driver's results are calculated independently.  
`orderBy("date", "raceId")` ensures that races are processed in chronological order.  
`rowsBetween(Window.unboundedPreceding, Window.currentRow)` allows cumulative aggregation from the first race up to the current race.

Using this window, I applied `sum()` to compute cumulative totals:
- `wins`
- `second_places`
- `third_places`

Finally, I joined the `drivers` dataset to include driver names, selected relevant columns, and used `.orderBy(col("wins").desc(), col("date").asc())` to sort the results first by the number of wins and then by race date.

6. Do drivers who make more pit stops perform better or worse?

I investigated whether the number of pit stops is related to race performance.

First, I calculated the number of pit stops each driver made in each race using the `pit_stops` dataset. Then, I joined this information with the `results` dataset to obtain each driver’s finishing position.

Since the analysis focuses on the relationship between pit stop frequency and performance, I used an inner join to retain only observations where pit stop data is available.

Finally, I sorted the results by finishing position in ascending order and pit stop count in descending order to make it easier to observe how pit stop frequency varies across different performance levels.

In [0]:
# Count pit stops per driver per race
df_pit_count = df_pitstops.select("raceId", "driverId", "stop").groupBy("raceId", "driverId").agg(count("stop").alias("pit_stop_count"))

# join with results
df_performance = df_results.select("raceId", "driverId", "positionOrder").join(df_pit_count, on=["raceId", "driverId"], how="inner").orderBy(col("positionOrder").asc(), col("pit_stop_count").desc())

display(df_performance)

First, I used `.groupBy("raceId", "driverId")` on the `pit_stops` dataset and applied `count("stop")` to calculate the number of pit stops each driver made in each race. This produces the `pit_stop_count` column.

Next, I selected the relevant columns (`raceId`, `driverId`, and `positionOrder`) from the `results` dataset and joined it with the aggregated pit stop table using `raceId` and `driverId`.

I used an inner join so that only driver-race combinations with available pit stop data are included in the analysis. This avoids null values and ensures that the comparison is meaningful.

Finally, I sorted the result using `.orderBy(col("positionOrder").asc(), col("pit_stop_count").desc())`, which arranges drivers from best to worst finishing position, and within the same position, places drivers with more pit stops first. This ordering helps reveal patterns between pit stop frequency and race performance.